In [1]:
import os
import json
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
import requests
from getMarketSalaryData import _getMarketSalaryData


In [2]:
url = 'https://api.openwebninja.com/job-salary-data/job-salary'
params = {
    'job_title': 'nodejs developer',
    'location': 'New York'
}

In [3]:
env_path = Path.cwd().parent / '.env'
if not env_path.exists():
    env_path = Path(find_dotenv())

load_dotenv(env_path, override=True)
api_key = os.getenv('OPENWEBNINJA_API_KEY', '').strip()
if not api_key:
    raise RuntimeError('OPENWEBNINJA_API_KEY was not found in .env')
if api_key.startswith('••'):
    raise RuntimeError('The API key is still masked. Paste the real key into .env, not the bullets from Postman.')

headers = {
    'x-api-key': api_key,
    'Accept': 'application/json'
}

In [4]:
response = requests.get(url, headers=headers, params=params, timeout=20)
response.raise_for_status()
data = response.json()
data

{'status': 'OK',
 'request_id': 'a26b4860-6632-4c41-8972-028f6d332c4e',
 'data': [{'location': 'New York City, NY',
   'job_title': 'Nodejs Developer',
   'min_salary': 118891.89,
   'max_salary': 197079.85,
   'median_salary': 152305.95,
   'min_base_salary': 97541.28,
   'max_base_salary': 157225.38,
   'median_base_salary': 123838.47,
   'min_additional_pay': 21350.61,
   'max_additional_pay': 39854.47,
   'median_additional_pay': 28467.48,
   'salary_period': 'YEAR',
   'salary_currency': 'USD',
   'salary_count': 7,
   'salaries_updated_at': '2025-04-10T23:59:59.000Z',
   'publisher_name': 'Glassdoor',
   'publisher_link': 'https://www.glassdoor.com/Salaries/company-salaries.htm?suggestCount=0&suggestChosen=false&sc.keyword=Nodejs%20Developer&locT=C&locId=1132348',
   'confidence': 'CONFIDENT'}]}

In [2]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [3]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

client = None
valid_api_key = False

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.isascii():
    bad_chars = [(index, f"U+{ord(char):04X}") for index, char in enumerate(api_key) if not char.isascii()]
    print(f"An API key was found, but it contains non-ASCII characters at: {bad_chars}. Please re-copy it from OpenAI and replace the value in your .env file.")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    client = OpenAI(api_key=api_key)
    valid_api_key = True
    print("API key found and looks good so far!")

API key found and looks good so far!


In [9]:
system_prompt= """
You're a senior HR assistant who have prior experience in negotiating salaries with Employees for FAANG companies.
You will be evaluating Salaries of employees based on their role, the city they work with respect to Market Salary cap based on their role and city they work in
and how likely the employee can switch due to the salary package.
The Employee's role is Nodejs Developer and location is New York whose current salary is 250,000.
Provide the evaluation description within 140 words. 
"""

In [10]:
user_prompt_prefix = """
Here's the Market Data for the role NodeJs Developer. The data is in the form of Json.
"""

In [11]:
def messages_for(data):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + json.dumps(data, indent=2)}
    ]

In [12]:
def _getSalaryEvaluation():
    if not valid_api_key:
        return "Skipping the OpenAI request until the API key is fixed."
    data = _getMarketSalaryData()
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(data)
    )
    return response.choices[0].message.content

In [13]:
_getSalaryEvaluation()

'The current salary of $250,000 for a Node.js Developer in New York City significantly exceeds the market max salary of around $197,080, as per recent Glassdoor data. The median salary is approximately $152,306, with additional compensation typically ranging between $21,351 and $39,854. This suggests the employee is very competitively compensated, likely benefiting from premium pay due to experience or specialized skills. Given this package surpasses market norms, the likelihood of the employee switching solely for salary reasons is low unless other companies offer exceptional benefits or equity stakes. Retention efforts should focus more on career development, work-life balance, and corporate culture rather than salary increments, as current compensation is already above industry standard for this role and location.'